In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics
#yolov8 is in this package only

In [ ]:
import os

dataset_path = '/content/drive/MyDrive/dataset'
print("Images Folders:", os.listdir(f"{dataset_path}/images"))
print("Labels Folders", os.listdir(f"{dataset_path}/labels"))

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/runs/drone_person/weights/last.pt') #loads a prev saved checkpoint, simce training got stopped a few times , so to resume from that epoch only

model.train(
    data='/content/drive/MyDrive/dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16 ,
    name='drone_person',
    project='/content/drive/MyDrive/runs',
    device=0,
    resume=True,
)

In [ ]:
!nvidia-smi

In [ ]:
from ultralytics import YOLO
import cv2

#load the best ckpt(best validation performance)
model = YOLO('/content/drive/MyDrive/runs/drone_person/weights/best.pt')

# Run on one test image
results = model.predict(
    source='/content/drive/MyDrive/dataset/images/uav0000305_00000_v/0000001.jpg',
    conf=0.1,
    save=True,
    project='/content/drive/MyDrive/test_output'
)

print(f"Detections: {len(results[0].boxes)}")

In [ ]:
results = model.predict(
    source='/content/drive/MyDrive/test_image.jpg',
    conf=0.1, #keeps detection above 10%
    save=True, #save o/p img with b.boxes
    project='/content/drive/MyDrive/test_output'
)
print(f"Detections: {len(results[0].boxes)}")

In [ ]:
import os

def count_lines(file_path): #this function counts lines inside a label file
    with open(file_path, "r", encoding="utf-8") as file:
        return sum(1 for line in file)

labels_root = "/content/drive/MyDrive/dataset/labels"
counts = {}

for seq in os.listdir(labels_root):
    seq_path = os.path.join(labels_root, seq)
    if not os.path.isdir(seq_path):
        continue
    total = 0
    for txt_file in os.listdir(seq_path):
        if txt_file.endswith(".txt"):
            full_path = os.path.join(seq_path, txt_file)
            total += count_lines(full_path)
    counts[seq] = total

for seq, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{seq}: {count} detections")

best_seq = max(counts, key=counts.get)
print(f"\nBest sequence: {best_seq}")